In [2]:
!pip install transformers tensorboard datasets peft trl accelerate evaluate bitsandbytes sentencepiece wandb unsloth

144.44s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
from huggingface_hub import logout
logout()


Not logged in!
[huggingface_hub._login|WARNING]Not logged in!


In [4]:
from huggingface_hub import notebook_login, HfApi
notebook_login()

In [5]:
from huggingface_hub import HfApi
try:
    api = HfApi() 
    user_info = api.whoami() 
    print(f"Token validated successfully! Logged in as: {user_info['name']}") 
except Exception as e:
    print(f"Token validation failed. Error: {e}")

Token validation failed. Error: Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.


In [6]:
import unsloth

from unsloth import FastLanguageModel

In [7]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

from trl import SFTTrainer

In [8]:
CONFIG = {

    "base_model":
        "meta-llama/Llama-3.1-8B-Instruct",

    "dataset_path":
        "./dataset.jsonl",

    "output_dir":
        "./outputs",

    "max_length":
        2048,

    "epochs":
        3,

    "learning_rate":
        2e-4,

    "batch_size":
        4,

    "gradient_accumulation":
        8,

    "seed":
        42,

    "lora_r":
        16,

    "lora_alpha":
        32,

    "lora_dropout":
        0.05
}

In [9]:
def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

In [10]:
print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))

props = torch.cuda.get_device_properties(0)

print(
    f"Memory: "
    f"{props.total_memory / 1024**3:.2f} GB"
)

True
AMD Instinct MI300X
Memory: 191.98 GB


In [11]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-telco-llm-chatbot-training-dataset")

print(ds)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'intent', 'category', 'tags', 'response'],
        num_rows: 26000
    })
})


In [12]:
unique_categories = ds["train"].unique("category")

print(unique_categories)
print(f"Total categories: {len(unique_categories)}")

['BILLING', 'COMPLAINTS', 'CONSUMPTION', 'CONTACT', 'PAYMENT', 'SERVICES', 'SUBSCRIPTION']
Total categories: 7


In [13]:
from datasets import ClassLabel

categories = sorted(set(ds["train"]["category"]))

class_label = ClassLabel(names=categories)

def encode_category(example):
    example["category"] = class_label.str2int(example["category"])
    return example

ds_encoded = ds["train"].map(encode_category)

ds_encoded = ds_encoded.cast_column(
    "category",
    class_label
)

In [14]:
dataset = ds_encoded.train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="category"
)

train_ds = dataset["train"]
val_ds = dataset["test"]

print(train_ds)
print(val_ds)

Dataset({
    features: ['instruction', 'intent', 'category', 'tags', 'response'],
    num_rows: 23400
})
Dataset({
    features: ['instruction', 'intent', 'category', 'tags', 'response'],
    num_rows: 2600
})


In [15]:
for i in range(5):
    sample = train_ds[i]

    print("=" * 80)
    print("INSTRUCTION:")
    print(sample["instruction"])

    print("\nINTENT:")
    print(sample["intent"])

    print("\nCATEGORY:")
    print(sample["category"])

    print("\nRESPONSE:")
    print(sample["response"])

INSTRUCTION:
i have to modify my fucking cell plan how cojld i do it

INTENT:
change_plan

CATEGORY:
6

RESPONSE:
To change your cell plan, please follow these steps:

1. Log in to your account on {{WEBSITE_URL}}.
2. Navigate to the {{PLAN_SECTION}} section.
3. Select the {{CHANGE_PLAN_OPTION}} option.
4. Review the available plans and choose the one that best suits your needs.
5. Confirm your selection and follow any prompts to complete the process.

If you encounter any issues, please contact customer support for further assistance.
INSTRUCTION:
need to see mobile payments how could i do it

INTENT:
check_mobile_payments

CATEGORY:
4

RESPONSE:
To verify your mobile payments, kindly follow the steps outlined below:

1. Launch the {{APP}} application on your smartphone.
2. Enter your login details to access your account.
3. Go to the {{PAYMENT_SECTION}} area.
4. Click on the {{PAYMENT_HISTORY_OPTION}} feature.
5. Examine your latest payment records.

If you require additional informat

In [16]:
for col in train_ds.column_names:
    missing = sum(
        x is None or str(x).strip() == ""
        for x in train_ds[col]
    )
    print(f"{col}: {missing}")

instruction: 0
intent: 0
category: 0
tags: 0
response: 0


In [17]:
response_lengths = [
    len(str(x).split())
    for x in train_ds["response"]
]

print("Min:", min(response_lengths))
print("Mean:", np.mean(response_lengths))
print("Median:", np.median(response_lengths))
print("95th percentile:", np.percentile(response_lengths, 95))
print("Max:", max(response_lengths))

Min: 46
Mean: 73.0875641025641
Median: 68.0
95th percentile: 109.0
Max: 171


In [18]:
instruction_lengths = [
    len(str(x).split())
    for x in train_ds["instruction"]
]

print("Min:", min(instruction_lengths))
print("Mean:", np.mean(instruction_lengths))
print("Median:", np.median(instruction_lengths))
print("95th percentile:", np.percentile(instruction_lengths, 95))
print("Max:", max(instruction_lengths))

Min: 2
Mean: 10.827820512820512
Median: 11.0
95th percentile: 15.0
Max: 23


In [19]:
from collections import Counter

intent_counts = Counter(train_ds["intent"])

print(f"Unique intents: {len(intent_counts)}")

for intent, count in intent_counts.most_common(20):
    print(intent, count)

Unique intents: 26
check_signal_coverage 913
report_problem 912
activate_phone 910
cancel_plan 908
report_poor_signal_coverage 907
human_agent 906
payment_methods 905
sign_up_for_plan 905
activate_roaming 905
pay 904
set_usage_limits 902
check_excess_data_charges 902
change_plan 901
invoices 901
dispute_invoice 899
check_cancellation_fee 898
check_usage 896
schedule_payments 896
deactivate_call_management_services 896
activate_call_management_services 896


In [20]:
from collections import Counter

print(Counter(train_ds["category"]))
print(Counter(val_ds["category"]))

Counter({5: 6300, 6: 4500, 4: 3600, 2: 2700, 1: 2700, 3: 1800, 0: 1800})
Counter({5: 700, 6: 500, 4: 400, 2: 300, 1: 300, 0: 200, 3: 200})


In [21]:
import re
from collections import Counter

placeholder_counter = Counter()

for response in train_ds["response"]:

    placeholders = re.findall(
        r"\{\{.*?\}\}",
        response
    )

    placeholder_counter.update(placeholders)

print("Unique placeholders:",
      len(placeholder_counter))

for ph, count in placeholder_counter.most_common():
    print(ph, count)

Unique placeholders: 55
{{WEBSITE_URL}} 21617
{{SERVICE_TYPE}} 3618
{{PAYMENT_SECTION}} 3600
{{NEW_PROVIDER}} 3552
{{SUPPORT_SECTION}} 2725
{{PLAN_SECTION}} 1806
{{SUPPORT_TEAM_CONTACT}} 1798
{{USAGE_SECTION}} 1798
{{MANAGEMENT_SERVICES_SECTION}} 1792
{{DEVICE_TYPE}} 1790
{{APP}} 1789
{{SIGNAL_COVERAGE_SECTION}} 913
{{SEARCH_BUTTON}} 913
{{REPORT_PROBLEM_OPTION}} 912
{{ACTIVATION_SECTION}} 910
{{CANCEL_SECTION}} 908
{{CANCEL_PLAN_OPTION}} 908
{{REPORT_POOR_SIGNAL_COVERAGE_OPTION}} 907
{{HUMAN_AGENT_OPTION}} 906
{{PAYMENT_METHODS_OPTION}} 905
{{COMPANY}} 905
{{SIGN_UP_BUTTON}} 905
{{ROAMING_SECTION}} 905
{{INTERNATIONAL_ROAMING_OPTION}} 905
{{EDIT_OPTION}} 902
{{DATA_SECTION}} 902
{{CHECK_EXCESS_DATA_CHARGES_OPTION}} 902
{{INFORMATION_DATA_CHARGES_SECTION}} 902
{{CHANGE_PLAN_OPTION}} 901
{{BILLING_SECTION}} 901
{{VIEW_PAST_BILLS}} 901
{{INVOICE_SECTION}} 899
{{DISPUTE_INVOICE_OPTION}} 899
{{DAYS_NUMBER}} 899
{{MY_ACCOUNT_SECTION}} 898
{{FEE_OPTION}} 898
{{CANCELLATION_FEES_SUBSECTION}} 

In [22]:
df = train_ds.to_pandas()

print(
    "Duplicate instructions:",
    df["instruction"].duplicated().sum()
)

print(
    "Duplicate responses:",
    df["response"].duplicated().sum()
)

Duplicate instructions: 13
Duplicate responses: 16535


In [23]:
print(
    "Unique responses:",
    len(set(train_ds["response"]))
)

print(
    "Total rows:",
    len(train_ds)
)

Unique responses: 6865
Total rows: 23400


In [24]:
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-8B",
    max_seq_length=512,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    AMD Instinct MI300X. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.2.4.git3d3aa833. ROCm Toolkit: 7.2.53211. Triton: 3.6.0+rocm7.2.4.git4ed88892
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


TimeoutError: Unsloth: HuggingFace seems to be down after trying for 120 seconds :(
Check https://status.huggingface.co/ for more details.
As a temporary measure, use modelscope with the same model name ie:
```
pip install modelscope
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
from unsloth import FastLanguageModel
model = FastLanguageModel.from_pretrained('unsloth/gpt-oss-20b')
```

In [ ]:
samples = [
    "{{WEBSITE_URL}}",
    "{{PAYMENT_SECTION}}",
    "{{SUPPORT_TEAM_CONTACT}}",
]

for s in samples:
    print("=" * 50)
    print(s)

    tokens = tokenizer.tokenize(s)

    print(tokens)
    print("count:", len(tokens))

In [ ]:
def format_example(example):
    return (
        f"User: {example['instruction']}\n"
        f"Assistant: {example['response']}"
    )

In [ ]:
lengths = []

for sample in train_ds.select(range(1000)):
    text = format_example(sample)

    token_count = len(
        tokenizer(
            text,
            add_special_tokens=True
        )["input_ids"]
    )

    lengths.append(token_count)

print("Min:", min(lengths))
print("Mean:", np.mean(lengths))
print("95th:", np.percentile(lengths,95))
print("Max:", max(lengths))

In [ ]:
from collections import defaultdict
from datasets import enable_progress_bar

enable_progress_bar()
mapping = defaultdict(set)

for row in train_ds:
    mapping[row["instruction"]].add(row["response"])

conflicts = {
    k:v for k,v in mapping.items()
    if len(v) > 1
}

print("Conflicting instructions:", len(conflicts))

for k,v in list(conflicts.items())[:5]:
    print("="*80)
    print(k)
    print("Response count:", len(v))

In [ ]:
responses_per_intent = {}

for intent in set(train_ds["intent"]):

    responses = set()

    for row in train_ds:
        if row["intent"] == intent:
            responses.add(row["response"])

    responses_per_intent[intent] = len(responses)

for intent, count in sorted(
    responses_per_intent.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(intent, count)

In [ ]:
def format_example(example):
    return {
        "text":
        f"<|im_start|>user\n"
        f"{example['instruction']}\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"{example['response']}\n"
        f"<|im_end|>"
    }

train_formatted = train_ds.map(format_example)
val_formatted = val_ds.map(format_example)

In [ ]:
print(train_formatted[0]["text"])

In [ ]:
print(tokenizer.chat_template)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",

        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
module_names = set()

for name, _ in model.named_modules():
    module_names.add(name.split(".")[-1])

for name in sorted(module_names):
    print(name)

In [ ]:
model.print_trainable_parameters()

In [ ]:
for name, module in model.named_modules():
    if "q_proj" in name:
        print(name)
        print(type(module))
        break

In [ ]:
for name, module in model.named_modules():
    if "lora_A" in name:
        print(name)
        break

In [ ]:
def format_example(example):
    return {
        "text":
        f"<|im_start|>user\n"
        f"{example['instruction']}\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"{example['response']}\n"
        f"<|im_end|>"
    }

train_formatted = train_ds.map(format_example)
val_formatted = val_ds.map(format_example)

In [ ]:
def format_chat(example):
    # Standard ChatML formatting without the empty reflection blocks
    text = (
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['response']}<|im_end|>"
    )
    return {"text": text}

# Re-map your datasets cleanly
train_formatted = train_ds.map(format_chat)
val_formatted = val_ds.map(format_chat)

In [ ]:

train_formatted = train_ds.map(format_chat)
val_formatted = val_ds.map(format_chat)

In [ ]:
print(train_formatted[0]["text"])

In [ ]:
import transformers
import peft
import unsloth

print(transformers.__version__)
print(peft.__version__)
print(unsloth.__version__)

In [ ]:
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None
prompt = tokenizer.apply_chat_template(
    [
        {
            "role": "user",
            "content": "i wanna change my internet plan"
        }
    ],
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.1
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

In [ ]:
lengths = []

for sample in train_formatted.select(range(5000)):
    lengths.append(
        len(
            tokenizer(
                sample["text"]
            )["input_ids"]
        )
    )

print("Mean:", np.mean(lengths))
print("95th:", np.percentile(lengths,95))
print("Max:", max(lengths))

In [ ]:
training_args = TrainingArguments(
    output_dir="./telco-qwen",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    weight_decay=0.01,

    warmup_ratio=0.03,

    lr_scheduler_type="cosine",

    logging_steps=20,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,

    bf16=True,

    report_to="wandb",

    packing = True,                   # <-- Pack short sequences together
    gradient_checkpointing = True, 
    fp16 = False,
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_formatted,
    eval_dataset = val_formatted,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = training_args,
)

trainer_output = trainer.train()